# 02 — Burgers' Equation

**A nonlinear PDE with shock formation — the classic benchmark from Raissi et al. (2019).**

The 1D viscous Burgers' equation on $x \in [-1, 1]$, $t \in [0, 1]$:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x}
- \nu\frac{\partial^2 u}{\partial x^2} = 0$$

$$u(0, x) = -\sin(\pi x), \qquad u(t, -1) = u(t, 1) = 0$$

For small $\nu$ the convection term $u\,u_x$ dominates and the smooth sine steepens into a
**near-discontinuous shock at $x = 0$** by $t \approx 0.4$ — a regime where classical solvers
need shock-capturing machinery. We use the benchmark viscosity $\nu = 0.01/\pi$.

> **Requirements** — run `uv sync --all-packages` at the repo root first, and start Jupyter
> from the project environment (`uv run jupyter lab`) so that the `pinn` library is importable.


## Recap: How the PINN Solves This

The PINN minimises a **weighted multi-term cost function**

$$
\mathcal{L}_{total} = w_{ic}\,\mathcal{L}_{ic} + w_{bc}\,\mathcal{L}_{bc} + w_{physics}\,\mathcal{L}_{physics}
$$

via the standard five-step loop: **guess** (forward pass at collocation points) →
**physics check** (exact derivatives via `torch.autograd.grad`, plugged into the PDE residual) →
**loss** (mean squared violations) → **correction** (backprop through the residual itself) →
**iterate**. See `01_harmonic_analysis.ipynb` for the full deep-dive on this mechanism.


## 1. Setup

In [ ]:
import numpy as np
import torch
import torch.autograd as autograd
import matplotlib.pyplot as plt

from pinn.core.network import PINN
from pinn.trainer.trainer import PINNTrainer
from pinn.utils.plotting import plot_contour

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Problem Configuration

> **Note:** 30k full-batch epochs over 5000 collocation points is a long run on CPU.
> For a quick smoke test drop `EPOCHS` to ~5000 — the shock will be smeared but visible.


In [ ]:
# --- Physical parameters ---
NU = 0.01 / np.pi          # viscosity (benchmark value)
X_DOMAIN = (-1.0, 1.0)
T_DOMAIN = (0.0, 1.0)

# --- Network / training hyperparameters ---
HIDDEN_LAYERS = 5
HIDDEN_NEURONS = 50
EPOCHS = 30_000
LR = 1e-3
N_IC, N_BC, N_PHYSICS = 100, 50, 5000
LOSS_WEIGHTS = {"ic": 1.0, "bc": 1.0, "physics": 1.0}

print(f"Burgers: u_t + u*u_x - {NU:.5f}*u_xx = 0")

## 3. Collocation Points and Loss Terms

Three constraint families (this problem needs no data loss — it is a pure forward problem):

| Term | Enforces | Points |
|------|----------|--------|
| `ic` | $u(0,x) = -\sin(\pi x)$ | 100 uniform in $x$ |
| `bc` | $u(t,\pm 1) = 0$ | 50 uniform in $t$ |
| `physics` | $u_t + u u_x - \nu u_{xx} = 0$ | 5000 **uniform-random** interior points |

Random interior sampling (vs. a grid) is standard for PINNs — it avoids aliasing artifacts and
covers the space-time domain economically.


In [ ]:
x_ic = torch.linspace(*X_DOMAIN, N_IC).view(-1, 1).to(device)
t_bc = torch.linspace(*T_DOMAIN, N_BC).view(-1, 1).to(device)

x_physics = (torch.rand(N_PHYSICS, 1) * (X_DOMAIN[1] - X_DOMAIN[0]) + X_DOMAIN[0])
t_physics = (torch.rand(N_PHYSICS, 1) * (T_DOMAIN[1] - T_DOMAIN[0]) + T_DOMAIN[0])
x_physics = x_physics.to(device).requires_grad_(True)
t_physics = t_physics.to(device).requires_grad_(True)


def pde_residual(model, x, t):
    xt = torch.cat([x, t], dim=1)
    u = model(xt)
    u_t = autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    u_x = autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_xx = autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    return u_t + u * u_x - NU * u_xx


def ic_loss(model):
    xt = torch.cat([x_ic, torch.zeros_like(x_ic)], dim=1)
    u = model(xt)
    u_exact = -torch.sin(np.pi * x_ic)
    return torch.mean((u - u_exact) ** 2)


def bc_loss(model):
    u_left = model(torch.cat([-torch.ones_like(t_bc), t_bc], dim=1))
    u_right = model(torch.cat([torch.ones_like(t_bc), t_bc], dim=1))
    return torch.mean(u_left**2 + u_right**2)


def physics_loss(model):
    return torch.mean(pde_residual(model, x_physics, t_physics) ** 2)

## 4. Model and Training

A plain `tanh` MLP, $(x, t) \mapsto u$ — no Ansatz needed here. The solution is $O(1)$-frequency;
the difficulty is the *sharp gradient*, which the 5×50 network plus the diffusive $\nu u_{xx}$
term can represent.


In [ ]:
model = PINN(input_dim=2, hidden_layers=HIDDEN_LAYERS, hidden_neurons=HIDDEN_NEURONS)
n_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {n_params}")

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
trainer = PINNTrainer(model, device=device)

trainer.train(
    n_epochs=EPOCHS,
    optimizer=optimizer,
    loss_functions={"ic": ic_loss, "bc": bc_loss, "physics": physics_loss},
    weights=LOSS_WEIGHTS,
    plot_every=0,
    debug_every=0,
)

In [ ]:
trainer.plot_loss_history(show_total=True)

## 5. Results: Space-Time Solution

The contour plot shows $u(t, x)$ over the whole domain. Look for the sharp colour transition
along $x = 0$ for $t \gtrsim 0.4$ — that is the shock.


In [ ]:
n_x, n_t = 200, 200
x_test = torch.linspace(*X_DOMAIN, n_x).view(-1, 1).to(device)
t_test = torch.linspace(*T_DOMAIN, n_t).view(-1, 1).to(device)
X, T = torch.meshgrid(x_test.squeeze(), t_test.squeeze(), indexing="ij")

with torch.no_grad():
    xt_test = torch.stack([X.flatten(), T.flatten()], dim=1)
    u_pred = model(xt_test).cpu().numpy().reshape(n_x, n_t)

plot_contour(
    X.cpu().numpy(), T.cpu().numpy(), u_pred,
    title="PINN Solution for Burgers' Equation",
    xlabel="t", ylabel="x", clabel="u(t,x)",
)

## 6. Validation Snapshots

- **$t = 0$:** must reproduce the known IC $-\sin(\pi x)$ — a direct accuracy check.
- **$t = 1$:** the fully-formed shock. No cheap closed form exists at low $\nu$, so we inspect
  the profile qualitatively: a steep, centred transition with flat wings.


In [ ]:
with torch.no_grad():
    u_pinn_0 = model(torch.cat([x_test, torch.zeros_like(x_test)], dim=1)).cpu().numpy()
    u_pinn_1 = model(torch.cat([x_test, torch.ones_like(x_test)], dim=1)).cpu().numpy()

x_np = x_test.cpu().numpy()
u_exact_0 = -np.sin(np.pi * x_np)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(x_np, u_exact_0, "k-", label="Exact (t=0)", linewidth=2)
axes[0].plot(x_np, u_pinn_0, "r--", label="PINN (t=0)", linewidth=2)
axes[0].set(title="Snapshot at t = 0", xlabel="x", ylabel="u(0,x)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(x_np, u_pinn_1, "r-", label="PINN (t=1)", linewidth=2)
axes[1].set(title="Snapshot at t = 1 (shock formed)", xlabel="x", ylabel="u(1,x)")
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

rel_l2_ic = np.linalg.norm(u_pinn_0 - u_exact_0) / np.linalg.norm(u_exact_0)
print(f"Relative L2 error at t=0 : {rel_l2_ic:.4e}")
print(f"Final total loss         : {trainer.loss_history[-1]['total']:.4e}")

## 7. Takeaways

1. **Equal loss weights work here** (unlike the harmonic oscillator) because all three terms are
   naturally $O(1)$ — weight tuning is problem-dependent, not a universal recipe.
2. **The shock is the stress test.** If it comes out smeared: train longer, or sample collocation
   points more densely near $x=0$ where the residual is hardest to satisfy.
3. **Mesh-free matters.** 5000 random points replace the careful grid + shock-capturing schemes
   a classical solver would need.

**Next:** `03_schrodinger_analysis.ipynb` — complex-valued fields and periodic boundary
conditions. The equivalent CLI run is `uv run train-burgers` (see `experiments/burgers/`).
